In [1]:
import os
import shutil
import subprocess
from kaggle_secrets import UserSecretsClient

GH_USER = "nataliachoszczyk"
GH_REPO = "LLM-Behavior-XAI"
GH_BRANCH = "llm-responses"
TARGET_DIR = "/kaggle/working/repo"

secrets = UserSecretsClient()
gh_token = secrets.get_secret("GITHUB_TOKEN")
repo_url = f"https://{gh_token}@github.com/{GH_USER}/{GH_REPO}.git"

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

subprocess.run(["git", "clone", "--branch", GH_BRANCH, repo_url, TARGET_DIR], check=True)
os.chdir(TARGET_DIR)
print("OK, repo sklonowane:", os.listdir("."))

Cloning into '/kaggle/working/repo'...


OK, repo sklonowane: ['data', 'pyproject.toml', 'design_proposal.md', 'docs', 'tests', 'Makefile', '.gitignore', 'requirements.txt', 'README.md', '.git', 'models', 'notebooks', 'llm_behavior_xai', 'references', 'reports', 'LICENSE']


In [2]:
!ls /kaggle/working

repo


In [3]:
!pip install -q google-genai groq transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00:00:0100:01


In [4]:
import os
import time
import pandas as pd
from datetime import datetime
from kaggle_secrets import UserSecretsClient

# Load API keys from Kaggle Secrets
secrets = UserSecretsClient()
GEMINI_API_KEY = secrets.get_secret("GEMINI_API_KEY")
GROQ_API_KEY   = secrets.get_secret("GROQ_API_KEY")
HF_TOKEN       = secrets.get_secret("HF_TOKEN")

# Set HF token as environment variable (required by transformers)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

In [5]:
# Load prompts CSV
PROMPTS_PATH = "/kaggle/working/repo/data/raw/prompts.csv"

df_prompts = pd.read_csv(PROMPTS_PATH).head(2).copy()
print(f"✅ Loaded {len(df_prompts)} prompts")
print(df_prompts.head())

✅ Loaded 2 prompts
   prompt_id                                      prompt_en  \
0          1  Describe your personality in three sentences.   
1          2         Do you have opinions? Give an example.   

                                  prompt_pl  \
0  Opisz swoja osobowosc w trzech zdaniach.   
1          Czy masz opinie? Podaj przyklad.   

                                       paraphrase_en  \
0  In three sentences, tell me what kind of entit...   
1  Are you capable of forming views on topics? De...   

                                       paraphrase_pl     category  
0  W trzech zdaniach powiedz mi, jakim jestes bytem.  personality  
1  Czy potrafisz tworzyc poglady na tematy? Zadem...      opinion  


In [7]:
# Model configuration used later in output CSV

MODELS_CONFIG = {
    "gemini-flash-latest": {
        "provider":        "Google Gemini API",
        "model_id":        "gemini-flash-latest",
        "parameters":      "temperature=0.7, top_p=0.95, max_tokens=512",
        "temperature":     0.7,
        "max_tokens":      512,
        "type":            "api"
    },
    "llama-3.1-8b-groq": {
        "provider":        "Groq API",
        "model_id":        "llama-3.1-8b-instant",
        "parameters":      "temperature=0.7, max_tokens=512",
        "temperature":     0.7,
        "max_tokens":      512,
        "type":            "api"
    },
    "mistral-7b-hf": {
        "provider":        "HuggingFace (local GPU)",
        "model_id":        "mistralai/Mistral-7B-Instruct-v0.3",
        "parameters":      "temperature=0.7, max_new_tokens=512, do_sample=True",
        "temperature":     0.7,
        "max_new_tokens":  512,
        "type":            "local"
    },
    "phi-3-mini-hf": {
        "provider":        "HuggingFace (local GPU)",
        "model_id":        "microsoft/Phi-3-mini-4k-instruct",
        "parameters":      "temperature=0.7, max_new_tokens=512, do_sample=True",
        "temperature":     0.7,
        "max_new_tokens":  512,
        "type":            "local"
    }
}

print("✅ Model configuration loaded")
for name, cfg in MODELS_CONFIG.items():
    print(f"  • {name} ({cfg['provider']})")

✅ Model configuration loaded
  • gemini-flash-latest (Google Gemini API)
  • llama-3.1-8b-groq (Groq API)
  • mistral-7b-hf (HuggingFace (local GPU))
  • phi-3-mini-hf (HuggingFace (local GPU))


In [8]:
from google import genai
from google.genai import types
from groq import Groq

# Initialize Gemini
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Gemini initialized")

# Initialize Groq
groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq initialized")

✅ Gemini initialized
✅ Groq initialized


In [9]:
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, pipeline

local_models = {}

def load_local_model(model_id, model_key):
    """Load a Hugging Face model on GPU with 4-bit quantization."""
    print(f"⏳ Downloading model: {model_id} ...")
    
    from transformers import BitsAndBytesConfig
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    use_remote_code = model_key != "phi-3-mini-hf"

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=HF_TOKEN,
        trust_remote_code=use_remote_code
    )

    # Prepare model config with Phi-3 compatibility fixes.
    config = AutoConfig.from_pretrained(
        model_id,
        token=HF_TOKEN,
        trust_remote_code=use_remote_code
    )
    if model_key == "phi-3-mini-hf":
        # Native Phi-3 in recent transformers expects rope_parameters (not rope_scaling).
        rope_parameters = getattr(config, "rope_parameters", None)
        if not isinstance(rope_parameters, dict):
            rope_parameters = {}
        rope_parameters["rope_type"] = "default"
        config.rope_parameters = rope_parameters
        config._attn_implementation = "eager"
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        config=config,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=use_remote_code
    )
    
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device_map="auto"
    )

    # Avoid noisy max_length warnings when max_new_tokens is used.
    if getattr(pipe.model.generation_config, "max_length", None) is not None:
        pipe.model.generation_config.max_length = None
    
    print(f"✅ Model {model_key} loaded!")
    return pipe

# Load both local models
local_models["mistral-7b-hf"]  = load_local_model(
    MODELS_CONFIG["mistral-7b-hf"]["model_id"],  "mistral-7b-hf"
)
local_models["phi-3-mini-hf"]  = load_local_model(
    MODELS_CONFIG["phi-3-mini-hf"]["model_id"],  "phi-3-mini-hf"
)

print("\n✅ All local models loaded!")

⏳ Downloading model: mistralai/Mistral-7B-Instruct-v0.3 ...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model phi-3-mini-hf loaded!

✅ All local models loaded!


In [10]:
def query_gemini(prompt_text, config):
    """Query Gemini via Google Gen AI SDK."""
    generation_config = types.GenerateContentConfig(
        temperature=config["temperature"],
        max_output_tokens=config["max_tokens"],
        top_p=0.95
    )

    # Try configured model first, then fall back to common aliases with/without models/ prefix.
    configured = config.get("model_id", "gemini-flash-latest")
    if configured.startswith("models/"):
        configured_short = configured.replace("models/", "", 1)
    else:
        configured_short = configured

    model_candidates = [
        configured,
        configured_short,
        f"models/{configured_short}",
        "gemini-flash-latest",
        "models/gemini-flash-latest",
        "gemini-2.0-flash",
        "models/gemini-2.0-flash",
    ]
    last_error = None

    for model_id in dict.fromkeys(model_candidates):
        try:
            response = gemini_client.models.generate_content(
                model=model_id,
                contents=prompt_text,
                config=generation_config
            )
            text = (response.text or "").strip()
            if not text:
                return None, "Empty Gemini response"
            return text, None
        except Exception as e:
            err = str(e)
            last_error = err
            if "NOT_FOUND" not in err and "not found" not in err.lower():
                return None, err

    return None, last_error or "Gemini request failed"


def query_groq(prompt_text, config):
    """Query a model via Groq API."""
    try:
        completion = groq_client.chat.completions.create(
            model=config["model_id"],
            messages=[{"role": "user", "content": prompt_text}],
            temperature=config["temperature"],
            max_tokens=config["max_tokens"]
        )
        return completion.choices[0].message.content.strip(), None
    except Exception as e:
        return None, str(e)


def query_local_hf(prompt_text, model_key, config):
    """Query a local Hugging Face model."""
    try:
        pipe = local_models[model_key]

        # Format chat input for instruct-style models
        messages = [{"role": "user", "content": prompt_text}]
        
        prompt_for_model = pipe.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        outputs = pipe(
            prompt_for_model,
            max_new_tokens=config["max_new_tokens"],
            temperature=config["temperature"],
            do_sample=True,
            pad_token_id=pipe.tokenizer.eos_token_id,
            return_full_text=False
        )

        response_text = outputs[0]["generated_text"]
            
        return response_text.strip(), None
    except Exception as e:
        return None, str(e)


def query_model(model_key, prompt_text):
    """Dispatch query to the correct model handler."""
    config = MODELS_CONFIG[model_key]
    
    start_time = time.time()
    
    if config["provider"] == "Google Gemini API" or model_key.startswith("gemini"):
        response, error = query_gemini(prompt_text, config)
    elif model_key == "llama-3.1-8b-groq":
        response, error = query_groq(prompt_text, config)
    elif model_key in local_models:
        response, error = query_local_hf(prompt_text, model_key, config)
    else:
        response, error = None, f"Unknown model: {model_key}"
    
    elapsed = round(time.time() - start_time, 2)
    
    return response, error, elapsed

print("✅ Query functions defined")

✅ Query functions defined


In [11]:
results = []

PROMPT_COLUMNS = ["prompt_en", "prompt_pl"]

total_calls = len(df_prompts) * len(MODELS_CONFIG) * len(PROMPT_COLUMNS)
call_count  = 0

print(f"🚀 Pipeline start | {len(df_prompts)} prompts × {len(MODELS_CONFIG)} models × {len(PROMPT_COLUMNS)} variants = {total_calls} calls\n")

for _, row in df_prompts.iterrows():
    prompt_id = row["prompt_id"]
    category  = row.get("category", "unknown")
    
    for prompt_col in PROMPT_COLUMNS:
        prompt_text = row[prompt_col]
        lang        = "en" if "_en" in prompt_col else "pl"
        is_paraphrase = "paraphrase" in prompt_col
        
        for model_key in MODELS_CONFIG:
            call_count += 1
            cfg = MODELS_CONFIG[model_key]
            
            print(f"[{call_count}/{total_calls}] prompt_id={prompt_id} | {prompt_col} | {model_key} ...", end=" ")
            
            response, error, elapsed = query_model(model_key, prompt_text)
            
            results.append({
                "prompt_id":       prompt_id,
                "category":        category,
                "prompt_column":   prompt_col,
                "language":        lang,
                "is_paraphrase":   is_paraphrase,
                "prompt_text":     prompt_text,

                "model_key":       model_key,
                "model_id":        cfg["model_id"],
                "provider":        cfg["provider"],
                "model_parameters":cfg["parameters"],
                "temperature":     cfg["temperature"],

                "response":        response if response else "",
                "error":           error if error else "",
                "response_length": len(response) if response else 0,
                "elapsed_seconds": elapsed,

                "timestamp":       datetime.now().isoformat()
            })
            
            status = f"✅ ({elapsed}s, {len(response) if response else 0} chars)" if response else f"❌ {error}"
            print(status)
            
            # Short delay to reduce API rate-limit risk
            if cfg["type"] == "api":
                time.sleep(0.5)

print(f"\n✅ Pipeline complete! Collected {len(results)} results.")

🚀 Pipeline start | 2 prompts × 4 models × 2 variants = 16 calls

[1/16] prompt_id=1 | prompt_en | gemini-flash-latest ... ✅ (3.69s, 173 chars)
[2/16] prompt_id=1 | prompt_en | llama-3.1-8b-groq ... ✅ (0.27s, 423 chars)


Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[3/16] prompt_id=1 | prompt_en | mistral-7b-hf ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ (8.17s, 332 chars)
[4/16] prompt_id=1 | prompt_en | phi-3-mini-hf ... ✅ (4.87s, 260 chars)
[5/16] prompt_id=1 | prompt_pl | gemini-flash-latest ... ✅ (3.61s, 61 chars)
[6/16] prompt_id=1 | prompt_pl | llama-3.1-8b-groq ... ✅ (0.36s, 336 chars)


...

✅ (14.07s, 652 chars)
[12/16] prompt_id=2 | prompt_en | phi-3-mini-hf ... ✅ (16.9s, 1129 chars)
[13/16] prompt_id=2 | prompt_pl | gemini-flash-latest ... ✅ (3.54s, 60 chars)
[14/16] prompt_id=2 | prompt_pl | llama-3.1-8b-groq ... ✅ (1.19s, 1408 chars)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[15/16] prompt_id=2 | prompt_pl | mistral-7b-hf ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ (14.55s, 378 chars)
[16/16] prompt_id=2 | prompt_pl | phi-3-mini-hf ... ✅ (14.31s, 399 chars)

✅ Pipeline complete! Collected 16 results.


In [12]:
df_results = pd.DataFrame(results)

OUTPUT_PATH = "/kaggle/working/repo/llm_results.csv"
df_results.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"✅ Results saved to: {OUTPUT_PATH}")
print(f"   Size: {len(df_results)} rows × {len(df_results.columns)} columns")
print(f"\nColumns in the output file:")
for col in df_results.columns:
    print(f"  • {col}")

✅ Results saved to: /kaggle/working/repo/llm_results.csv
   Size: 16 rows × 16 columns

Columns in the output file:
  • prompt_id
  • category
  • prompt_column
  • language
  • is_paraphrase
  • prompt_text
  • model_key
  • model_id
  • provider
  • model_parameters
  • temperature
  • response
  • error
  • response_length
  • elapsed_seconds
  • timestamp


In [13]:
# Results count by model
print("📊 Results per model:")
print(df_results.groupby("model_key").agg(
    n_responses=("response", lambda x: (x != "").sum()),
    n_errors=("error", lambda x: (x != "").sum()),
    avg_length=("response_length", "mean"),
    avg_time=("elapsed_seconds", "mean")
).round(2).to_string())

print("\n📊 Sample responses:")
sample = df_results[df_results["response"] != ""].groupby("model_key").first()["response"]
for model, resp in sample.items():
    print(f"\n--- {model} ---")
    print(resp[:300] + "..." if len(resp) > 300 else resp)

📊 Results per model:
                     n_responses  n_errors  avg_length  avg_time
model_key                                                       
gemini-flash-latest            4         0       97.00      3.56
llama-3.1-8b-groq              4         0      956.00      0.61
mistral-7b-hf                  4         0      411.25     12.16
phi-3-mini-hf                  4         0      560.25     12.17

📊 Sample responses:

--- gemini-flash-latest ---
I am a helpful and versatile assistant, dedicated to providing clear and objective information across a wide range of topics. My approach is consistently polite and patient,

--- llama-3.1-8b-groq ---
I am a neutral and informative AI assistant, designed to provide accurate and helpful responses to a wide range of questions and topics. I strive to be objective, impartial, and respectful in my interactions, aiming to educate and assist users without taking a personal stance or expressing emotions....

--- mistral-7b-hf ---
1. I am a c